In [2]:
import numpy as np
import random
import math
import time
import os
import matplotlib.cm as cm
import pandas as pd
import glob
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import matplotlib.pyplot as plt
%matplotlib inline

from tqdm.notebook import tqdm
from math import sqrt
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error

## Load data

In [3]:
Battery_list = ['CS2_35', 'CS2_36', 'CS2_37', 'CS2_38']
Battery = np.load('datasets/CALCE/CALCE.npy', allow_pickle=True)
Battery = Battery.item() # A dict with cell name as key and dataframe as value

In [4]:
Battery.keys()

dict_keys(['CS2_35', 'CS2_36', 'CS2_37', 'CS2_38'])

In [5]:
Battery["CS2_35"]

,cycle,capacity,SoH,resistance,CCCT,CVCT
0,1.0,1.126385,0.825175,0.094009,6613.059052,2251.498033
1,2.0,1.126160,0.815965,0.091661,6612.402800,2231.967052
2,3.0,1.125966,0.815977,0.094649,6608.560673,2228.216959
3,4.0,1.118508,0.825194,0.091413,6604.732222,2247.561061
4,5.0,1.117210,0.806900,0.091413,6629.211049,2077.692393
...,...,...,...,...,...,...
877,878.0,0.339218,0.128353,0.121454,1255.711238,NaN
878,879.0,0.330025,0.128343,0.122852,1535.380556,3328.912803
879,880.0,0.330027,0.119175,0.120863,1291.478395,3022.668035
880,881.0,0.320860,0.119175,0.122264,1290.025300,3007.262188


## EDA of this csv
https://chatgpt.com/c/67c35896-fe8c-8009-a4f6-818229b28590

## How does the paper calculate RE, MAE, and RMSE?

# Understand function from the repo

In [6]:
from utils import get_train_test
name = "CS2_35"
feature_size = 8

train_x, train_y, train_data, test_data = get_train_test(Battery, name, feature_size)

In [12]:
print(f'train_x shape: {train_x.shape}, train_y shape: {train_y.shape}, train_data len: {len(train_data)}, test_data len: {len(test_data)}')



train_x shape: (2881, 8), train_y shape: (2881,), train_data len: 9, test_data len: 873


In [13]:
train_data

[1.126384506847021,
 1.1261598161259705,
 1.125965708259707,
 1.1185076863180938,
 1.1172100749182547,
 1.1137535513150443,
 1.0988181436744804,
 1.0961803263765921,
 1.0898711301696573]

In [14]:
test_data

[1.0875822954661674,
 1.0947478949707463,
 1.0955682863426888,
 1.0954329180709088,
 1.095223222083344,
 1.0948040555332583,
 1.0944448817959302,
 1.0940442107208557,
 1.0934614215105336,
 1.0927332194072785,
 1.0888536787324268,
 1.0883814134478704,
 1.0879949624513225,
 1.0876290188457574,
 1.0872335492167664,
 1.086799750378054,
 1.0863976880895259,
 1.0764608398671511,
 1.0682813219671374,
 1.061871030040747,
 1.0604499542130021,
 1.0590703515513098,
 1.0684316315261961,
 1.069764576241007,
 1.0612467360899798,
 1.0528038388294765,
 1.0514969839157156,
 1.0507885373892558,
 1.049771042431389,
 1.05916552576209,
 1.0604562774048463,
 1.057632133197537,
 1.0482737033997178,
 1.043392804566336,
 1.0429167637310581,
 1.0420172556913927,
 1.0497244102947476,
 1.0519062232518257,
 1.056326663309842,
 1.0414672148233903,
 1.0410028838709957,
 1.0404851130044386,
 1.0392121440794764,
 1.0869982407359577,
 1.0806073900944038,
 1.0867494055373201,
 1.086558905233767,
 1.0774217859911137,
 1.

In [17]:
from utils import build_sequences


data_dict = Battery
window_size = feature_size
data_sequence=data_dict[name]['capacity']
data_sequence

0      1.126385
1      1.126160
2      1.125966
3      1.118508
4      1.117210
         ...   
877    0.339218
878    0.330025
879    0.330027
880    0.320860
881    0.320863
Name: capacity, Length: 882, dtype: float64

In [18]:
train_data, test_data = data_sequence[:window_size+1], data_sequence[window_size+1:]
train_data, test_data

(0    1.126385
 1    1.126160
 2    1.125966
 3    1.118508
 4    1.117210
 5    1.113754
 6    1.098818
 7    1.096180
 8    1.089871
 Name: capacity, dtype: float64,
 9      1.087582
 10     1.094748
 11     1.095568
 12     1.095433
 13     1.095223
          ...   
 877    0.339218
 878    0.330025
 879    0.330027
 880    0.320860
 881    0.320863
 Name: capacity, Length: 873, dtype: float64)

In [19]:

train_x, train_y = build_sequences(text=train_data, window_size=window_size)
train_x, train_y

(array([[1.12638451, 1.12615982, 1.12596571, 1.11850769, 1.11721007,
         1.11375355, 1.09881814, 1.09618033]]),
 array([1.08987113]))

In [20]:
for k, v in data_dict.items():
    if k != name:
        data_x, data_y = build_sequences(text=v['capacity'], window_size=window_size)
        print(f'k: {k}, data_x shape: {data_x.shape}, data_y shape: {data_y.shape}')
        train_x, train_y = np.r_[train_x, data_x], np.r_[train_y, data_y]
        print(f'train_x shape: {train_x.shape}, train_y shape: {train_y.shape}')

k: CS2_36, data_x shape: (928, 8), data_y shape: (928,)
train_x shape: (929, 8), train_y shape: (929,)
k: CS2_37, data_x shape: (964, 8), data_y shape: (964,)
train_x shape: (1893, 8), train_y shape: (1893,)
k: CS2_38, data_x shape: (988, 8), data_y shape: (988,)
train_x shape: (2881, 8), train_y shape: (2881,)
